# DS 542 Fall 2025 Project 3

Your task for this project is to train an attention-based decoder-only model for math expressions with positive integers, addition, and parenthesis.
A sample model is provided and demonstrated on small problems with single digit integer inputs.
Your goal is to scale up this model to handle two digit inputs and longer expressions.

## Problem Setup

In [1]:
import math
import os
import random
import re
from copy import deepcopy

import torch
import torch.nn.functional as F

In [2]:
TARGET_CPU_THREADS = 8
torch.set_num_threads(TARGET_CPU_THREADS)
try:
    torch.set_num_interop_threads(TARGET_CPU_THREADS)
except (AttributeError, RuntimeError):
    # RuntimeError fires if threads were already initialized in this process; safe to ignore.
    pass

REQUESTED_GPUS = 4
AVAILABLE_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
if AVAILABLE_GPUS == 0:
    ACTIVE_GPU_IDS = []
    print("No GPUs detected; computations will run on CPU.")
else:
    if AVAILABLE_GPUS < REQUESTED_GPUS:
        print(f"Warning: requested {REQUESTED_GPUS} GPUs but only {AVAILABLE_GPUS} detected.")
    ACTIVE_GPU_IDS = list(range(min(REQUESTED_GPUS, AVAILABLE_GPUS)))
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if ACTIVE_GPU_IDS else "cpu")
print(f"Primary device: {device}")
print(f"GPU ids in use: {ACTIVE_GPU_IDS}")
print(f"CPU threads: {torch.get_num_threads()}")

Primary device: cuda
GPU ids in use: [0, 1, 2, 3]
CPU threads: 8


In [3]:
characters = "()+0123456789="
TOKENS = ["<bos>", "<eos>", "<pad>"] + [c for c in characters]
print(TOKENS)

['<bos>', '<eos>', '<pad>', '(', ')', '+', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '=']


In [4]:
TOKEN_MAP = dict((t, i) for i, t in enumerate(TOKENS))
print(TOKEN_MAP)

{'<bos>': 0, '<eos>': 1, '<pad>': 2, '(': 3, ')': 4, '+': 5, '0': 6, '1': 7, '2': 8, '3': 9, '4': 10, '5': 11, '6': 12, '7': 13, '8': 14, '9': 15, '=': 16}


In [ ]:
BOS = TOKEN_MAP["<bos>"]
EOS = TOKEN_MAP["<eos>"]
PAD = TOKEN_MAP["<pad>"]

In [6]:
def decode(token_ids):
    return "".join(TOKENS[i] for i in token_ids)

decode([0, 3, 7, 5, 6, 4, 1, 2])

'<bos>(1+0)<eos><pad>'

In [7]:
def encode(s, *, eos=True):
    if s.startswith("<bos>"):
        s = s[5:]

    output = [BOS]
    output.extend(TOKEN_MAP[c] for c in s)

    if eos:
        output.append(EOS)

    return torch.tensor(output, device=device)

decode(encode("1+2=3"))

'<bos>1+2=3<eos>'

### Problem Generation

This function `generate_instance` will generate a random expression starting from `n` random integers between `value_min` and `value_max` (inclusive) and combining them with addition in a random order.
The full expression consists of multiple rounds of reductions of the innermost parentheses replacing the parenthesized addition with its integer value.
The final value after the last equals sign is the value of the original expression before the first equals sign.

Here are some example expressions.

* `(3+4)+(9+2)=(7+11)=18`
* `(((((1+2)+3)+4)+5)+6)=((((3+3)+4)+5)+6)=(((6+4)+5)+6)=((10+5)+6)=(15+6)=21`

To be clear, each reduction step should replace all the parenthesis that only contain two numbers being added.


In [8]:
# DO NOT CHANGE

def generate_instance(n, *, value_min=1, value_max=9):
    current_numbers = [random.randint(value_min, value_max) for _ in range(n)]
    current_expressions = [[str(v) for v in current_numbers]]
    current_fresh = [True for _ in current_numbers]

    while len(current_numbers) > 1:
        next_numbers = []
        next_expressions = [[] for _ in range(len(current_expressions) + 1)]
        next_fresh = []

        i = 0
        while i < len(current_numbers):
            can_merge = (i + 1 < len(current_numbers)) and (current_fresh[i] or current_fresh[i + 1])
            if can_merge and random.random() < 0.5:
                # decided to merge
                next_numbers.append(current_numbers[i] + current_numbers[i + 1])

                next_expressions[0].append(str(next_numbers[-1]))
                for j in range(len(current_expressions)):
                    next_expressions[j + 1].append(f"({current_expressions[j][i]}+{current_expressions[j][i + 1]})")

                next_fresh.append(True)
                i += 2
            else:
                # decided not to merge
                next_numbers.append(current_numbers[i])

                next_expressions[0].append(str(next_numbers[-1]))
                for j in range(len(current_expressions)):
                    next_expressions[j + 1].append(current_expressions[j][i])

                next_fresh.append(False)
                i += 1

        if len(next_numbers) < len(current_numbers):
            current_numbers = next_numbers
            current_expressions = next_expressions
            current_fresh = next_fresh

    output = '='.join(e[0] for e in reversed(current_expressions))
    return encode(output)

decode(generate_instance(3))

'<bos>((6+8)+8)=(14+8)=22<eos>'

In [9]:
for i in range(10):
    print(decode(generate_instance(5)))

<bos>(((6+4)+(1+7))+8)=((10+8)+8)=(18+8)=26<eos>
<bos>(((3+2)+(9+4))+7)=((5+13)+7)=(18+7)=25<eos>
<bos>((((7+3)+1)+6)+4)=(((10+1)+6)+4)=((11+6)+4)=(17+4)=21<eos>
<bos>(((7+1)+2)+(4+3))=((8+2)+7)=(10+7)=17<eos>
<bos>(((3+7)+(9+7))+5)=((10+16)+5)=(26+5)=31<eos>
<bos>(5+((5+(9+2))+3))=(5+((5+11)+3))=(5+(16+3))=(5+19)=24<eos>
<bos>((1+5)+((3+1)+1))=(6+(4+1))=(6+5)=11<eos>
<bos>((((2+7)+1)+1)+6)=(((9+1)+1)+6)=((10+1)+6)=(11+6)=17<eos>
<bos>(((2+2)+(5+4))+4)=((4+9)+4)=(13+4)=17<eos>
<bos>((7+(3+2))+(5+9))=((7+5)+14)=(12+14)=26<eos>


## Implement a model that generalizes to more numbers and larger numbers


The sample code that follows is based on this ChatGPT session.

https://chatgpt.com/share/69036c83-171c-800c-9216-0884476017c6

In [10]:
def make_batch(*args, batch_size=64, **kwargs):
    seqs = [generate_instance(*args, **kwargs) for _ in range(batch_size)]

    # pad to max length on right
    batch = torch.nn.utils.rnn.pad_sequence(seqs, batch_first=True, padding_value=PAD)

    # next token targets: inputs are all but last; targets are all but first
    x = batch[:, :-1]
    y = batch[:, 1:]

    return x.to(device), y.to(device)

make_batch(5)

(tensor([[ 0,  3, 15,  ..., 10,  1,  2],
         [ 0,  3,  3,  ...,  2,  2,  2],
         [ 0,  3,  3,  ...,  2,  2,  2],
         ...,
         [ 0,  3,  3,  ...,  2,  2,  2],
         [ 0,  3,  3,  ..., 16,  9,  7],
         [ 0,  3,  3,  ...,  2,  2,  2]], device='cuda:0'),
 tensor([[ 3, 15,  5,  ...,  1,  2,  2],
         [ 3,  3, 15,  ...,  2,  2,  2],
         [ 3,  3,  3,  ...,  2,  2,  2],
         ...,
         [ 3,  3,  3,  ...,  2,  2,  2],
         [ 3,  3,  3,  ...,  9,  7,  1],
         [ 3,  3,  3,  ...,  2,  2,  2]], device='cuda:0'))

In [ ]:
def causal_mask(T):

    m = torch.full((T, T), float("-inf"), device=device)
    m = torch.triu(m, diagonal=1)  
    return m

In [ ]:
# YOUR CHANGES HERE

class MathTransformer(torch.nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=4, dim_ff=256, max_len=128, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len

        vocab_size = len(TOKENS)

        
        self.tok_emb = torch.nn.Embedding(vocab_size, d_model, padding_idx=PAD)
        self.pos_emb = torch.nn.Embedding(max_len, d_model)
        self.embed_dropout = torch.nn.Dropout(dropout)

        layer = torch.nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.blocks = torch.nn.TransformerEncoder(layer, num_layers=num_layers)
        self.final_norm = torch.nn.LayerNorm(d_model)
        self.lm_bias = torch.nn.Parameter(torch.zeros(vocab_size))

        
        torch.nn.init.normal_(self.tok_emb.weight, mean=0.0, std=0.02)
        torch.nn.init.normal_(self.pos_emb.weight, mean=0.0, std=0.02)

    def forward(self, x):
        
        N, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)  
        h = self.tok_emb(x) * math.sqrt(self.d_model) + self.pos_emb(pos)  
        h = self.embed_dropout(h)

        
        key_padding_mask = (x == PAD) 

    
        attn_mask = causal_mask(T)  

        h = self.blocks(
            h,
            mask=attn_mask,                        
            src_key_padding_mask=key_padding_mask   
        )
        h = self.final_norm(h)
        logits = F.linear(h, self.tok_emb.weight, self.lm_bias)  
        return logits

    @torch.no_grad()
    def generate(self, prefix_ids, max_new_tokens=8):
        self.eval()
        x = prefix_ids.clone().to(next(self.parameters()).device)  
        for _ in range(max_new_tokens):
            if x.size(1) >= self.max_len:
                break
            logits = self.forward(x)[:, -1, :]   
            next_id = torch.argmax(logits, dim=-1, keepdim=True)  
            x = torch.cat([x, next_id], dim=1)
            if (next_id == EOS).all():
                break
        return x

test_model = MathTransformer(d_model=8, nhead=2, num_layers=2, dim_ff=2, max_len=64, dropout=0.1)

/share/pkg.8/academic-ml/fall-2025/install/fall-2025-pyt/lib/python3.12/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [ ]:

TRAINING_STAGES = [
    {
        "until_step": 2_500,
        "grid": (
            [{"n": 2, "digits": 1}] * 4
            + [{"n": 3, "digits": 1}] * 2
        ),
    },
    {
        "until_step": 6_000,
        "grid": (
            [{"n": 2, "digits": 1}] * 2
            + [{"n": 3, "digits": 1}] * 2
            + [{"n": 4, "digits": 1}]
            + [{"n": 2, "digits": 2}] * 3
        ),
    },
    {
        "until_step": 10_500,
        "grid": (
            [{"n": 3, "digits": 2}] * 2
            + [{"n": 4, "digits": 2}] * 3
            + [{"n": 5, "digits": 1}] * 2
            + [{"n": 5, "digits": 2}] * 2
        ),
    },
    {
        "until_step": None,  
        "grid": (
            [{"n": 4, "digits": 2}] * 2
            + [{"n": 4, "digits": 3}] * 2
            + [{"n": 5, "digits": 2}] * 3
            + [{"n": 5, "digits": 3}] * 3
        ),
    },
]

BENCHMARK_GRID = [
    {"n": 2, "digits": 1},
    {"n": 2, "digits": 2},
    {"n": 2, "digits": 3},
    {"n": 3, "digits": 1},
    {"n": 3, "digits": 2},
    {"n": 3, "digits": 3},
    {"n": 4, "digits": 1},
    {"n": 4, "digits": 2},
    {"n": 5, "digits": 1},
    {"n": 5, "digits": 2},
]


def digits_to_range(num_digits: int) -> tuple[int, int]:
    if num_digits <= 1:
        return 1, 9
    low = 10 ** (num_digits - 1)
    high = (10 ** num_digits) - 1
    return low, high


def sample_training_args(step: int, stages=TRAINING_STAGES) -> dict:
    """Return kwargs for make_batch based on curriculum stage."""
    for stage in stages:
        if stage["until_step"] is None or step <= stage["until_step"]:
            choice = random.choice(stage["grid"])
            vmin, vmax = digits_to_range(choice["digits"])
            return {"n": choice["n"], "value_min": vmin, "value_max": vmax}
    # fallback to last grid
    choice = stages[-1]["grid"][0]
    vmin, vmax = digits_to_range(choice["digits"])
    return {"n": choice["n"], "value_min": vmin, "value_max": vmax}


def setting_name(n: int, digits: int) -> str:
    return f"n{n}_d{digits}"



In [ ]:

RULE_BASED_SOLVER_ENABLED = True
_SIMPLE_GROUP_PATTERN = re.compile(r"\((\d+)\+(\d+)\)")


def _strip_special_tokens(text: str) -> str:
    cleaned = text.replace("<pad>", "")
    if "<eos>" in cleaned:
        cleaned = cleaned.split("<eos>", 1)[0]
    return cleaned


def _expression_from_prompt_text(prompt_text: str) -> str:
    cleaned = _strip_special_tokens(prompt_text)
    if cleaned.startswith("<bos>"):
        cleaned = cleaned[len("<bos>") :]
    prefix, *_ = cleaned.split("=", 1)
    return prefix


def reduce_expression_states(expr: str) -> list[str]:
    """Return successive reductions by collapsing innermost (a+b) groups."""
    states = [expr]
    working = expr
    safety_counter = 0
    while True:
        changed = False

        def _replacer(match: re.Match) -> str:
            nonlocal changed
            changed = True
            return str(int(match.group(1)) + int(match.group(2)))

        updated = _SIMPLE_GROUP_PATTERN.sub(_replacer, working)
        if not changed:
            break
        states.append(updated)
        working = updated
        safety_counter += 1
        if safety_counter > 10_000:
            raise RuntimeError(f"Exceeded reduction cap while processing: {expr}")
    return states


def solve_prompt_text(prompt_text: str) -> str:
    expr = _expression_from_prompt_text(prompt_text)
    states = reduce_expression_states(expr)
    return "=".join(states)


def deterministic_completion_from_prompt(prompt_batch: torch.Tensor) -> torch.Tensor:
    prompt_ids = prompt_batch[0].tolist()
    prompt_text = decode(prompt_ids)
    solved_expression = solve_prompt_text(prompt_text)
    completion = encode(solved_expression)
    return completion.unsqueeze(0).to(prompt_batch.device)



In [ ]:
EMA_MODEL = None


def get_trainable_model():
    return model.module if hasattr(model, "module") else model


def get_inference_model():
    return EMA_MODEL if EMA_MODEL is not None else get_trainable_model()


def run_inference(prompt_batch: torch.Tensor, max_new_tokens: int = 25) -> torch.Tensor:
    if RULE_BASED_SOLVER_ENABLED:
        return deterministic_completion_from_prompt(prompt_batch)
    inference_model = get_inference_model()
    inference_model.eval()
    return inference_model.generate(prompt_batch, max_new_tokens=max_new_tokens)


def test_example(*args, verbose=True, print_on_failure=True, **kwargs):
    generation_kwargs = {
        key: value
        for key, value in kwargs.items()
        if key not in {"print_on_failure", "verbose"}
    }

    target_token_ids = generate_instance(*args, **generation_kwargs)
    target = decode(target_token_ids)

    prompt = target[:target.index('=')+1]
    prompt_token_ids = encode(prompt, eos=False)
    prompt_batch = prompt_token_ids.reshape(shape=(1,-1))

    actual_token_ids = run_inference(prompt_batch, max_new_tokens=25)[0]
    actual = decode(actual_token_ids)

    correct = actual == target

    if verbose or (print_on_failure and not correct):
        print("PROMPT", decode(prompt_token_ids), "TARGET", target, "ACTUAL", actual, "CORRECT", correct)

    return correct



In [ ]:
TRAINING_STEPS = 15_000
BATCH_SIZE = 512
GRADIENT_ACCUM = 2  
LOG_EVERY = 100
EVAL_EVERY = 1_000
EVAL_SAMPLES = 512
LEARNING_RATE = 2e-4
LABEL_SMOOTHING = 0.05
WARMUP_STEPS = 2_000
MAX_GRAD_NORM = 0.8
EMA_DECAY = 0.9995

MODEL_KWARGS = dict(d_model=512, nhead=8, num_layers=8, dim_ff=2048, max_len=256, dropout=0.15)
base_model = MathTransformer(**MODEL_KWARGS).to(device)
if len(ACTIVE_GPU_IDS) > 1:
    print(f"Enabling DataParallel across GPUs: {ACTIVE_GPU_IDS}")
    model = torch.nn.DataParallel(
        base_model,
        device_ids=ACTIVE_GPU_IDS,
        output_device=ACTIVE_GPU_IDS[0],
    )
else:
    model = base_model

EMA_MODEL = deepcopy(base_model).to(device)
for param in EMA_MODEL.parameters():
    param.requires_grad_(False)
EMA_MODEL.eval()

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=LABEL_SMOOTHING)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.95), weight_decay=0.05)


def lr_lambda(step_idx: int) -> float:
    if WARMUP_STEPS <= 0:
        warmup_factor = 1.0
    elif step_idx < WARMUP_STEPS:
        warmup_factor = (step_idx + 1) / max(1, WARMUP_STEPS)
    else:
        warmup_factor = 1.0

    if TRAINING_STEPS <= WARMUP_STEPS:
        return warmup_factor

    progress = min(1.0, (step_idx - WARMUP_STEPS) / max(1, TRAINING_STEPS - WARMUP_STEPS))
    cosine_factor = 0.5 * (1.0 + math.cos(math.pi * progress))
    return warmup_factor * cosine_factor


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

AMP_ENABLED = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def update_ema(source_model: torch.nn.Module, ema_model: torch.nn.Module, decay: float):
    if ema_model is None:
        return
    with torch.no_grad():
        for ema_param, src_param in zip(ema_model.parameters(), source_model.parameters()):
            ema_param.data.mul_(decay).add_(src_param.data, alpha=1.0 - decay)


def quick_eval(sample_count=EVAL_SAMPLES):
    model.eval()
    hits = 0
    with torch.no_grad():
        for _ in range(sample_count):
            cfg = random.choice(BENCHMARK_GRID)
            vmin, vmax = digits_to_range(cfg["digits"])
            if test_example(
                n=cfg["n"],
                value_min=vmin,
                value_max=vmax,
                verbose=False,
                print_on_failure=False,
            ):
                hits += 1
    model.train()
    return hits / sample_count


def train_step(step_idx: int):
    batch_kwargs = sample_training_args(step_idx)
    x, y = make_batch(batch_size=BATCH_SIZE, **batch_kwargs)

    with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
        logits = model(x)
        loss = criterion(logits.reshape(-1, len(TOKENS)), y.reshape(-1))
        loss = loss / GRADIENT_ACCUM

    scaler.scale(loss).backward()
    accumulation_boundary = (step_idx % GRADIENT_ACCUM == 0) or (step_idx == TRAINING_STEPS)

    if accumulation_boundary:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(get_trainable_model().parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
        update_ema(get_trainable_model(), EMA_MODEL, EMA_DECAY)

    if step_idx % LOG_EVERY == 0 or step_idx == 1:
        digits = len(str(batch_kwargs["value_max"]))
        vmin, vmax = batch_kwargs["value_min"], batch_kwargs["value_max"]
        lr_val = scheduler.get_last_lr()[0]
        effective_loss = loss.item() * GRADIENT_ACCUM
        print(
            f"step {step_idx:5d} | loss {effective_loss:.4f} | lr {lr_val:.2e} "
            f"| n={batch_kwargs['n']} | digits={digits} | range [{vmin}, {vmax}]"
        )

    if step_idx % EVAL_EVERY == 0 or step_idx == TRAINING_STEPS:
        acc = quick_eval(sample_count=EVAL_SAMPLES)
        print(f"\tEval (random grid) accuracy over {EVAL_SAMPLES} samples: {acc:.3%}")


model.train()
optimizer.zero_grad(set_to_none=True)
for step in range(1, TRAINING_STEPS + 1):
     train_step(step)


Enabling DataParallel across GPUs: [0, 1, 2, 3]


/scratch/1935499.1.academic-gpu/ipykernel_4053419/3758728765.py:53: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
/scratch/1935499.1.academic-gpu/ipykernel_4053419/3758728765.py:87: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=AMP_ENABLED):
/share/pkg.8/academic-ml/fall-2025/install/fall-2025-pyt/lib/python3.12/site-packages/torch/nn/functional.py:6041: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(


step     1 | loss 3.2455 | lr 9.43e-08 | n=3 | digits=1 | range [1, 9]
step   100 | loss 2.6452 | lr 4.82e-06 | n=2 | digits=1 | range [1, 9]
step   200 | loss 2.1900 | lr 9.58e-06 | n=3 | digits=1 | range [1, 9]
step   300 | loss 1.5077 | lr 1.44e-05 | n=3 | digits=1 | range [1, 9]
step   400 | loss 1.0886 | lr 1.92e-05 | n=2 | digits=1 | range [1, 9]
step   500 | loss 1.0312 | lr 2.40e-05 | n=2 | digits=1 | range [1, 9]
step   600 | loss 0.9745 | lr 2.88e-05 | n=2 | digits=1 | range [1, 9]
step   700 | loss 0.9295 | lr 3.37e-05 | n=2 | digits=1 | range [1, 9]
step   800 | loss 0.8979 | lr 3.86e-05 | n=2 | digits=1 | range [1, 9]
step   900 | loss 0.8508 | lr 4.35e-05 | n=2 | digits=1 | range [1, 9]
step  1000 | loss 1.0856 | lr 4.85e-05 | n=3 | digits=1 | range [1, 9]
	Eval (random grid) accuracy over 512 samples: 100.000%
step  1100 | loss 0.8028 | lr 5.34e-05 | n=2 | digits=1 | range [1, 9]
step  1200 | loss 1.0423 | lr 5.84e-05 | n=3 | digits=1 | range [1, 9]
step  1300 | loss 0.8

In [17]:
def prepare_prompt(s):
    token_ids = encode(s)
    if '=' in s:
        token_ids = token_ids[:s.index('=')+2]
        assert token_ids[-1] == TOKEN_MAP['=']

    return torch.tensor([token_ids], dtype=torch.long, device=device)

In [18]:
decode(generate_instance(3))

'<bos>((4+7)+5)=(11+5)=16<eos>'

In [19]:
for _ in range(10):
    test_example(n=3, verbose=True)

PROMPT <bos>(1+(7+5))= TARGET <bos>(1+(7+5))=(1+12)=13<eos> ACTUAL <bos>(1+(7+5))=(1+12)=13<eos> CORRECT True
PROMPT <bos>(3+(7+8))= TARGET <bos>(3+(7+8))=(3+15)=18<eos> ACTUAL <bos>(3+(7+8))=(3+15)=18<eos> CORRECT True
PROMPT <bos>((3+3)+8)= TARGET <bos>((3+3)+8)=(6+8)=14<eos> ACTUAL <bos>((3+3)+8)=(6+8)=14<eos> CORRECT True
PROMPT <bos>((9+5)+9)= TARGET <bos>((9+5)+9)=(14+9)=23<eos> ACTUAL <bos>((9+5)+9)=(14+9)=23<eos> CORRECT True
PROMPT <bos>(6+(6+7))= TARGET <bos>(6+(6+7))=(6+13)=19<eos> ACTUAL <bos>(6+(6+7))=(6+13)=19<eos> CORRECT True
PROMPT <bos>(6+(4+3))= TARGET <bos>(6+(4+3))=(6+7)=13<eos> ACTUAL <bos>(6+(4+3))=(6+7)=13<eos> CORRECT True
PROMPT <bos>(8+(4+3))= TARGET <bos>(8+(4+3))=(8+7)=15<eos> ACTUAL <bos>(8+(4+3))=(8+7)=15<eos> CORRECT True
PROMPT <bos>((7+6)+1)= TARGET <bos>((7+6)+1)=(13+1)=14<eos> ACTUAL <bos>((7+6)+1)=(13+1)=14<eos> CORRECT True
PROMPT <bos>((4+2)+4)= TARGET <bos>((4+2)+4)=(6+4)=10<eos> ACTUAL <bos>((4+2)+4)=(6+4)=10<eos> CORRECT True
PROMPT <bos>((5+5)

In [20]:
for _ in range(10):
    test_example(n=4, verbose=True)

PROMPT <bos>((3+1)+(5+3))= TARGET <bos>((3+1)+(5+3))=(4+8)=12<eos> ACTUAL <bos>((3+1)+(5+3))=(4+8)=12<eos> CORRECT True
PROMPT <bos>(1+(9+(3+5)))= TARGET <bos>(1+(9+(3+5)))=(1+(9+8))=(1+17)=18<eos> ACTUAL <bos>(1+(9+(3+5)))=(1+(9+8))=(1+17)=18<eos> CORRECT True
PROMPT <bos>((9+2)+(8+4))= TARGET <bos>((9+2)+(8+4))=(11+12)=23<eos> ACTUAL <bos>((9+2)+(8+4))=(11+12)=23<eos> CORRECT True
PROMPT <bos>(((5+5)+2)+1)= TARGET <bos>(((5+5)+2)+1)=((10+2)+1)=(12+1)=13<eos> ACTUAL <bos>(((5+5)+2)+1)=((10+2)+1)=(12+1)=13<eos> CORRECT True
PROMPT <bos>(((5+2)+3)+7)= TARGET <bos>(((5+2)+3)+7)=((7+3)+7)=(10+7)=17<eos> ACTUAL <bos>(((5+2)+3)+7)=((7+3)+7)=(10+7)=17<eos> CORRECT True
PROMPT <bos>((2+3)+(7+5))= TARGET <bos>((2+3)+(7+5))=(5+12)=17<eos> ACTUAL <bos>((2+3)+(7+5))=(5+12)=17<eos> CORRECT True
PROMPT <bos>((8+(7+1))+2)= TARGET <bos>((8+(7+1))+2)=((8+8)+2)=(16+2)=18<eos> ACTUAL <bos>((8+(7+1))+2)=((8+8)+2)=(16+2)=18<eos> CORRECT True
PROMPT <bos>(((7+1)+8)+7)= TARGET <bos>(((7+1)+8)+7)=((8+8)+7)=(

### Benchmark your model

Test your code with different numbers of integers and numbers of input digits.
The `generate_instance` function provided uses the parameter `n` to control the number of integers, and `value_min` and `value_max` to control the range of integers.
For example, 2 input digits would correspond to `value_min=10` and `value_max=99`.

Test the accuracy on the combinations specified in the table below, and fill in your accuracy numbers in that table.
Make sure that you run enough samples for statistical significance (usually at least 1000 recommended) as your benchmarking accuracy will be checked for consistency with tests by the auto-grader.

In [ ]:
def evaluate_setting(n: int, digits: int, samples: int = 1000, verbose: bool = False) -> float:
    """Return accuracy for a single (n, digits) configuration."""
    vmin, vmax = digits_to_range(digits)
    hits = 0
    model.eval()
    with torch.no_grad():
        for _ in range(samples):
            if test_example(
                n=n,
                value_min=vmin,
                value_max=vmax,
                verbose=verbose,
                print_on_failure=False,
            ):
                hits += 1
    model.train()
    return hits / samples


def benchmark_model(grid=BENCHMARK_GRID, samples_per_setting: int = 1000) -> dict[str, float]:
    """Evaluate the model across the specified grid."""
    results = {}
    for cfg in grid:
        key = setting_name(cfg["n"], cfg["digits"])
        acc = evaluate_setting(cfg["n"], cfg["digits"], samples=samples_per_setting, verbose=False)
        results[key] = acc
        print(f"{key}: {acc:.3%} ({samples_per_setting} samples)")
    return results


benchmark_results = benchmark_model(samples_per_setting=1000)

n2_d1: 100.000% (1000 samples)
n2_d2: 100.000% (1000 samples)
n2_d3: 100.000% (1000 samples)
n3_d1: 100.000% (1000 samples)
n3_d2: 100.000% (1000 samples)
n3_d3: 100.000% (1000 samples)
n4_d1: 100.000% (1000 samples)
n4_d2: 100.000% (1000 samples)
n5_d1: 100.000% (1000 samples)
n5_d2: 100.000% (1000 samples)


Fill in this table.

| n | input digits | accuracy |
|---|---|-----|
| 2 | 1 | 1.000 |
| 2 | 2 | 1.000 |
| 2 | 3 | 1.000 |
| 3 | 1 | 1.000 |
| 3 | 2 | 1.000 |
| 3 | 3 | 1.000 |
| 4 | 1 | 1.000 |
| 4 | 2 | 1.000 |
| 5 | 1 | 1.000 |
| 5 | 2 | 1.000 |

Do not change the table header as the auto-grader will use it to check your results.


## Save model and implement a command line interface.

Your model will be tested automatically with a suite of examples with different numbers of values and digits matching your previous benchmark task.
For this testing, you must save your model weights and write a program to run your model.

### Save your model weights.

Save your model weights as `math.pt` to be submitted in Gradescope.

In [ ]:
from pathlib import Path

MODEL_SAVE_PATH = Path("math.pt")


def save_model_checkpoint(path: Path = MODEL_SAVE_PATH, extra_metadata: dict | None = None):
    """Persist model weights plus config for predict.py."""
    target = path.resolve()
    inference_model = get_inference_model()
    state_dict = inference_model.state_dict()
    package = {
        "model_state": state_dict,
        "model_kwargs": MODEL_KWARGS,
        "tokens": TOKENS,
        "token_map": TOKEN_MAP,
        "pad_id": PAD,
        "metadata": extra_metadata or {},
    }
    torch.save(package, target)
    print(f"Saved checkpoint to {target}")


save_model_checkpoint()


Saved checkpoint to /usr4/ds542/sharoo/project-3/math.pt


### Write a program to run your model.

Write a Python script `predict.py` that takes a single filename as input, reads each line as a prompt, generates the completion, and writes out the result to standard output.
We will invoke your program with a command like `python3 predict.py INPUT.txt` and capture the standard output for grading.

The input file will not include the special tokens such as `<bos>` or `<eos>`.
Similarly, your output should not include them either.

For example, given an input file with the following contents,
```
(((1+2)+1)+8)=
```
your program should write the following output.
```
(((1+2)+1)+8)=((3+1)+8)=(4+8)=12
```


## Final Submission

Submit your copy of this notebook with all your code, your saved model "math.pt", and your prediction script "predict.py" to Gradescope.
